# Transfer learning runs (Member 3)

Experiment notebook: it only calls the project's CLI commands, all code lives in `src/`.

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (about 1–1.5 hours for the three runs)

Every run is copied to Google Drive (`MyDrive/Deep-Learning-transfer/`) as soon as it finishes,
so a disconnect does not lose finished runs. At the end, `transfer_results_small.zip`
(CSV/JSON/PNG only, no checkpoints) is downloaded for committing to GitHub.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Google Drive (trained models are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Deep-Learning-transfer'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print('Results will be saved to:', DRIVE_DIR)

## 2. Project code and dependencies

In [ ]:
!git clone -q https://github.com/joumana-wael/Deep-Learning.git /content/Deep-Learning
%cd /content/Deep-Learning
!git log --oneline -1
!pip install -q datasets mlflow

In [ ]:
# Offline checks of the transfer model (shapes, freezing, BatchNorm, overfit)
!python test_transfer_model.py

In [ ]:
import shutil
from pathlib import Path

def save_to_drive(path):
    """Copy a run folder (or file) from the Colab disk to Google Drive."""
    path = Path(path)
    target = Path(DRIVE_DIR) / path.name
    if path.is_dir():
        shutil.copytree(path, target, dirs_exist_ok=True)
    elif path.exists():
        shutil.copy2(path, target)
    print('Saved to Drive:', target)

## 3. Training
Same split, augmentation and seed as the scratch model; ImageNet normalization.

### ResNet-50 — frozen

In [ ]:
!python -m src.training.train_transfer --config configs/transfer_frozen.json --num-workers 2
save_to_drive('outputs/transfer_resnet50_frozen')
save_to_drive('mlflow.db')

### ResNet-50 — partial

In [ ]:
!python -m src.training.train_transfer --config configs/transfer_partial.json --num-workers 2
save_to_drive('outputs/transfer_resnet50_partial')
save_to_drive('mlflow.db')

### ResNet-50 — finetune

In [ ]:
!python -m src.training.train_transfer --config configs/transfer_finetune.json --num-workers 2
save_to_drive('outputs/transfer_resnet50_finetune')
save_to_drive('mlflow.db')

## 4. Scratch vs transfer comparison

In [ ]:
!python -m src.training.compare_models     --run scratch=outputs/scratch_cnn_v2     --run frozen=outputs/transfer_resnet50_frozen     --run partial=outputs/transfer_resnet50_partial     --run finetune=outputs/transfer_resnet50_finetune
save_to_drive('outputs/comparison')

In [ ]:
from IPython.display import Image, Markdown, display

display(Markdown(Path('outputs/comparison/comparison_table.md').read_text()))
for plot in ['best_accuracy.png', 'accuracy_curves.png', 'loss_curves.png']:
    display(Image(f'outputs/comparison/{plot}'))

## 5. Download the small results for GitHub
Checkpoints (`best_model.pth`) stay on Drive only.

In [ ]:
import zipfile

run_dirs = [
    'outputs/transfer_resnet50_frozen',
    'outputs/transfer_resnet50_partial',
    'outputs/transfer_resnet50_finetune',
    'outputs/comparison',
]

zip_path = 'transfer_results_small.zip'
with zipfile.ZipFile(zip_path, 'w') as archive:
    for run_dir in run_dirs:
        for file in Path(run_dir).rglob('*'):
            if file.suffix in {'.csv', '.json', '.png', '.md'}:
                archive.write(file)
                print('added', file)

save_to_drive(zip_path)

from google.colab import files
files.download(zip_path)